# Mode choice ML 

## 1. Data Preparation

In [1]:
# 1. Libraries and project configuration

from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import json

# Project directories
project_path = Path(
    r"D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect"
)

data_path = project_path / "data" / "analytical"
mnl_path = project_path / "outputs" / "MNLM_models"
ml_path = project_path / "outputs" / "ML_models"

ml_path.mkdir(parents=True, exist_ok=True)

print("Project:", project_path)
print("Analytical data:", data_path)
print("MNL results:", mnl_path)
print("ML outputs:", ml_path)

Project: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect
Analytical data: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\data\analytical
MNL results: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\outputs\MNLM_models
ML outputs: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\outputs\ML_models


In [2]:
# 2. Load MNL results and data partitions

with open(mnl_path / "MNL_complete_results_backup.pkl", "rb") as file:
    mnl_results = pickle.load(file)

with open(mnl_path / "MNL_B6_model_configuration.json", "r", encoding="utf-8") as file:
    b6_config = json.load(file)

partitions = np.load(
    mnl_path / "MNL_data_partitions_and_CV_folds.npz"
)

# Confirm final MNL specification
print("Final MNL model:", b6_config["final_model"])
print("B6 parameters:", b6_config["b6_expected_parameters"])

# Confirm saved partitions
print("\nTraining observations:", len(partitions["train_indices"]))
print("Test observations:", len(partitions["test_indices"]))
print("CV folds:", b6_config["validation"]["n_splits"])

Final MNL model: B6
B6 parameters: 251

Training observations: 80114
Test observations: 20029
CV folds: 5


In [3]:
# 3. Load analytical dataset

data_file = data_path / "df_base.csv"

df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five observations:")
display(df.head())

Dataset shape: (100143, 19)

Columns:
['key_viaje', 'modo_principal_agrupado', 'modo_principal_desagrupado', 'edad', 'sexo', 'income_group', 'estra_hg', 'vehicle_availability', 'has_car_license', 'has_motorcycle_license', 'has_public_transport_license', 'purpose_group', 'time_period_6', 'zat_centroid_distance_m', 'travel_frequency_3', 'education_group', 'occupation_group', 'household_size_group', 'duracion_min']

First five observations:


C:\Users\ASUS\AppData\Local\Temp\ipykernel_19096\400156243.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_file)


,key_viaje,modo_principal_agrupado,modo_principal_desagrupado,edad,sexo,income_group,estra_hg,vehicle_availability,has_car_license,has_motorcycle_license,has_public_transport_license,purpose_group,time_period_6,zat_centroid_distance_m,travel_frequency_3,education_group,occupation_group,household_size_group,duracion_min
0,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,AUTO,Vehículo privado como conductor,41,Hombre,NS_NR,3,2+,1,0,0,WORK,AM_PEAK,12730.630894,DAILY,SECONDARY,EMPLOYEE,4_5_PERSONS,60.0
1,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,AUTO,Vehículo privado como conductor,41,Hombre,NS_NR,3,2+,1,0,0,HOME_RETURN,PM_PEAK,12730.630894,DAILY,SECONDARY,EMPLOYEE,4_5_PERSONS,95.0
2,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,BICICLETA,Bicicleta convencional como conductor,42,Mujer,NS_NR,3,2+,0,0,0,WORK,AM_PEAK,1454.544478,DAILY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,30.0
3,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,BICICLETA,Bicicleta convencional como conductor,42,Mujer,NS_NR,3,2+,0,0,0,HOME_RETURN,PM_OFF_PEAK,1454.544478,DAILY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,3.0
4,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,A PIE > 15 MIN,A pie (Mayores de 15 min),42,Mujer,NS_NR,3,2+,0,0,0,PERSONAL_ERRANDS,PM_OFF_PEAK,1134.070694,RARELY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,15.0


In [4]:
# 4. Validate original train/test partitions

train_idx = partitions["train_indices"]
test_idx = partitions["test_indices"]

print("Dataset rows:", len(df))
print("Maximum saved index:", max(train_idx.max(), test_idx.max()))
print("Train/test overlap:", len(np.intersect1d(train_idx, test_idx)))

# Recover original partitions
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

# Verify target alignment
print("\nTarget alignment:")
print("Train:", np.array_equal(
    df_train["modo_principal_agrupado"].to_numpy(),
    partitions["y_train"]
))
print("Test:", np.array_equal(
    df_test["modo_principal_agrupado"].to_numpy(),
    partitions["y_test"]
))

print("\nPartition shapes:")
print("Train:", df_train.shape)
print("Test:", df_test.shape)

Dataset rows: 100143
Maximum saved index: 100142
Train/test overlap: 0

Target alignment:
Train: False
Test: False

Partition shapes:
Train: (80114, 19)
Test: (20029, 19)


In [5]:
# 5. Inspect target encoding

print("Dataset target:")
print(df["modo_principal_agrupado"].value_counts(dropna=False))

print("\nSaved MNL y_train:")
print(pd.Series(partitions["y_train"]).value_counts(dropna=False))

print("\nSaved MNL y_test:")
print(pd.Series(partitions["y_test"]).value_counts(dropna=False))

print("\nData types:")
print("Dataset:", df["modo_principal_agrupado"].dtype)
print("MNL y_train:", partitions["y_train"].dtype)

Dataset target:
modo_principal_agrupado
TRANSPORTE PÚBLICO    29400
A PIE > 15 MIN        24923
AUTO                  12582
A PIE <15 MIN         12387
MOTO                   6247
BICICLETA              6101
TAXI OCUPADO           3266
TRANSPORTE ESCOLAR     2058
ESPECIAL OCUPADO       1981
INFORMAL                866
OTRO                    332
Name: count, dtype: int64

Saved MNL y_train:
10    23520
2     19939
0     10066
1      9909
6      4998
3      4881
8      2612
9      1646
4      1585
5       693
7       265
Name: count, dtype: int64

Saved MNL y_test:
10    5880
2     4984
0     2516
1     2478
6     1249
3     1220
8      654
9      412
4      396
5      173
7       67
Name: count, dtype: int64

Data types:
Dataset: object
MNL y_train: int8


In [6]:
# 6. Reconstruct and validate target encoding

target_classes = sorted(df["modo_principal_agrupado"].dropna().unique())

target_mapping = {
    label: code
    for code, label in enumerate(target_classes)
}

y_train_check = (
    df_train["modo_principal_agrupado"]
    .map(target_mapping)
    .to_numpy()
)

y_test_check = (
    df_test["modo_principal_agrupado"]
    .map(target_mapping)
    .to_numpy()

)

print("Target mapping:")
print(target_mapping)

print("\nTrain alignment:",
      np.array_equal(y_train_check, partitions["y_train"]))

print("Test alignment:",
      np.array_equal(y_test_check, partitions["y_test"]))

Target mapping:
{'A PIE <15 MIN': 0, 'A PIE > 15 MIN': 1, 'AUTO': 2, 'BICICLETA': 3, 'ESPECIAL OCUPADO': 4, 'INFORMAL': 5, 'MOTO': 6, 'OTRO': 7, 'TAXI OCUPADO': 8, 'TRANSPORTE ESCOLAR': 9, 'TRANSPORTE PÚBLICO': 10}

Train alignment: False
Test alignment: False


In [7]:
# 7. Reconstruct the original MNL target encoding

target_classes = [
    "AUTO",
    "A PIE <15 MIN",
    "A PIE > 15 MIN",
    "BICICLETA",
    "ESPECIAL OCUPADO",
    "INFORMAL",
    "MOTO",
    "OTRO",
    "TAXI OCUPADO",
    "TRANSPORTE ESCOLAR",
    "TRANSPORTE PÚBLICO"
]

target_mapping = {
    label: code
    for code, label in enumerate(target_classes)
}

y_train_check = df_train["modo_principal_agrupado"].map(target_mapping).to_numpy()
y_test_check = df_test["modo_principal_agrupado"].map(target_mapping).to_numpy()

print("Train alignment:",
      np.array_equal(y_train_check, partitions["y_train"]))

print("Test alignment:",
      np.array_equal(y_test_check, partitions["y_test"]))

Train alignment: True
Test alignment: True


In [8]:
# 8. Define ML features and target

b3_features = [
    "has_car_license",
    "has_motorcycle_license",
    "sexo",
    "estra_hg",
    "vehicle_availability",
    "purpose_group",
    "time_period_6",
    "edad",
    "zat_centroid_distance_m"
]

target = "modo_principal_agrupado"

X_train = df_train[b3_features].copy()
X_test = df_test[b3_features].copy()

y_train = partitions["y_train"].copy()
y_test = partitions["y_test"].copy()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)
print("\nMissing values in training features:")
print(X_train.isna().sum())

Training features: (80114, 9)
Test features: (20029, 9)
Training target: (80114,)
Test target: (20029,)

Missing values in training features:
has_car_license            0
has_motorcycle_license     0
sexo                       0
estra_hg                   0
vehicle_availability       0
purpose_group              0
time_period_6              0
edad                       0
zat_centroid_distance_m    0
dtype: int64


In [9]:
# 9. Inspect feature data types and categories

print("Feature data types:")
print(X_train.dtypes)

print("\nCategorical variables:")
for col in X_train.select_dtypes(include=["object", "category"]).columns:
    print(f"\n{col}:")
    print(X_train[col].value_counts(dropna=False))

print("\nNumerical variables:")
print(X_train.select_dtypes(include=np.number).describe().T)

Feature data types:
has_car_license              int64
has_motorcycle_license       int64
sexo                        object
estra_hg                    object
vehicle_availability        object
purpose_group               object
time_period_6               object
edad                         int64
zat_centroid_distance_m    float64
dtype: object

Categorical variables:

sexo:
sexo
Mujer          41594
Hombre         38507
Intersexual       13
Name: count, dtype: int64

estra_hg:
estra_hg
2            20636
No aplica    18066
3            17510
1            11165
4             5681
5             3249
6             2358
3             1113
4              182
6               80
5               74
Name: count, dtype: int64

vehicle_availability:
vehicle_availability
0     32059
1     31730
2+    16325
Name: count, dtype: int64

purpose_group:
purpose_group
HOME_RETURN         38135
WORK                16824
PERSONAL_ERRANDS     8225
EDUCATION            7507
ESCORT               3261
HEALT

### Preprocessmiento

In [11]:
# 10. Normalize categorical variables and classify features

# Standardize socioeconomic stratum as categorical
for dataset in [X_train, X_test]:
    dataset["estra_hg"] = dataset["estra_hg"].astype(str)

# Identify binary, categorical and numerical features
binary_features = []
categorical_features = []
numerical_features = []

for col in b3_features:
    unique_values = X_train[col].dropna().unique()

    if len(unique_values) == 2:
        binary_features.append(col)

    elif pd.api.types.is_numeric_dtype(X_train[col]):
        numerical_features.append(col)

    else:
        categorical_features.append(col)

print("BINARY FEATURES:", binary_features)
print("CATEGORICAL FEATURES:", categorical_features)
print("NUMERICAL FEATURES:", numerical_features)

# Validate feature classification
classified_features = (
    binary_features + categorical_features + numerical_features
)

assert len(classified_features) == len(b3_features)
assert len(set(classified_features)) == len(b3_features)

BINARY FEATURES: ['has_car_license', 'has_motorcycle_license']
CATEGORICAL FEATURES: ['sexo', 'estra_hg', 'vehicle_availability', 'purpose_group', 'time_period_6']
NUMERICAL FEATURES: ['edad', 'zat_centroid_distance_m']


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("binary", "passthrough", binary_features),
        ("categorical", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ), categorical_features),
        ("numerical", StandardScaler(), numerical_features)
    ],
    remainder="drop"
)

print("Preprocessor successfully defined.")

Preprocessor successfully defined.


## 2. Model Development Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Define Random Forest baseline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest baseline successfully defined.")

Random Forest baseline successfully defined.


In [14]:
# Load the original cross-validation folds
cv_folds = [
    (
        partitions[f"fold_{i}_train"],
        partitions[f"fold_{i}_validation"]
    )
    for i in range(1, 6)
]

# Check fold sizes
for i, (train_idx, val_idx) in enumerate(cv_folds, start=1):
    print(
        f"Fold {i}: "
        f"Training = {len(train_idx):,}, "
        f"Validation = {len(val_idx):,}"
    )

Fold 1: Training = 64,091, Validation = 16,023
Fold 2: Training = 64,090, Validation = 16,024
Fold 3: Training = 64,091, Validation = 16,023
Fold 4: Training = 64,092, Validation = 16,022
Fold 5: Training = 64,092, Validation = 16,022


In [15]:
# Validate cross-validation indices
for i, (train_idx, val_idx) in enumerate(cv_folds, start=1):

    assert train_idx.max() < len(X_train)
    assert val_idx.max() < len(X_train)
    assert len(np.intersect1d(train_idx, val_idx)) == 0

    print(f"Fold {i}: indices validated.")

print("\nAll five folds are valid.")

Fold 1: indices validated.
Fold 2: indices validated.
Fold 3: indices validated.
Fold 4: indices validated.
Fold 5: indices validated.

All five folds are valid.


In [16]:
from sklearn.model_selection import cross_validate

# Define evaluation metrics
scoring = {
    "macro_f1": "f1_macro",
    "log_loss": "neg_log_loss"
}

# Run 5-fold cross-validation
rf_cv_results = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv_folds,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False
)

# Display fold results
rf_cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": rf_cv_results["test_macro_f1"],
    "Log Loss": -rf_cv_results["test_log_loss"]
})

display(rf_cv_summary)

,Fold,Macro-F1,Log Loss
0,1,0.325846,2.165650
1,2,0.332326,2.069369
2,3,0.332326,2.098085
3,4,0.327659,2.138395
4,5,0.334063,2.069981


### 2.1. Importancia de Variables

In [17]:
# Fit Random Forest on the complete training set
rf_model.fit(X_train, y_train)

print("Random Forest fitted on the complete training set.")

Random Forest fitted on the complete training set.


In [18]:
# Extract feature importances
classifier = rf_model.named_steps["classifier"]
encoder = rf_model.named_steps["preprocessor"].named_transformers_["categorical"]

importances = classifier.feature_importances_

# Aggregate one-hot encoded features by original variable
feature_importance = {}

for feature in binary_features + numerical_features:
    feature_importance[feature] = importances[
        rf_model.named_steps["preprocessor"].output_indices_[
            "binary" if feature in binary_features else "numerical"
        ].start + (
            binary_features.index(feature)
            if feature in binary_features
            else numerical_features.index(feature)
        )
    ]

categorical_names = encoder.get_feature_names_out(categorical_features)
categorical_start = rf_model.named_steps["preprocessor"].output_indices_["categorical"].start

for feature in categorical_features:
    positions = [
        categorical_start + i
        for i, name in enumerate(categorical_names)
        if name.startswith(feature + "_")
    ]
    feature_importance[feature] = importances[positions].sum()

# Create ranked table
rf_importance = (
    pd.DataFrame({
        "Variable": feature_importance.keys(),
        "Importance": feature_importance.values()
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(rf_importance)

,Variable,Importance
0,zat_centroid_distance_m,0.337018
1,edad,0.310731
2,time_period_6,0.079454
3,estra_hg,0.075404
4,purpose_group,0.060336
5,has_motorcycle_license,0.044907
6,vehicle_availability,0.043415
7,has_car_license,0.025567
8,sexo,0.023169


### 2.2 Optimizacion Random froest

In [19]:
# Define a compact Random Forest hyperparameter search
rf_configs = [
    {"max_depth": depth, "min_samples_leaf": leaf, "max_features": "sqrt"}
    for depth in [15, 25]
    for leaf in [5, 10]
]

for i, config in enumerate(rf_configs, start=1):
    print(f"Configuration {i}: {config}")

Configuration 1: {'max_depth': 15, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
Configuration 2: {'max_depth': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
Configuration 3: {'max_depth': 25, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
Configuration 4: {'max_depth': 25, 'min_samples_leaf': 10, 'max_features': 'sqrt'}


In [21]:
# Reload original partitions, including object arrays
partitions = np.load(
    mnl_path / "MNL_data_partitions_and_CV_folds.npz",
    allow_pickle=True
)

groups_train = partitions["groups_train"]

print("Groups loaded:", len(groups_train))
print("Training observations:", len(X_train))

Groups loaded: 80114
Training observations: 80114


In [22]:
from sklearn.model_selection import StratifiedGroupKFold

# Recover original training groups
groups_train = partitions["groups_train"]

# Create a single stratified, grouped validation split
preselection_cv = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=42
)

preselection_train_idx, preselection_val_idx = next(
    preselection_cv.split(X_train, y_train, groups_train)
)

print(f"Training: {len(preselection_train_idx):,}")
print(f"Validation: {len(preselection_val_idx):,}")

Training: 68,669
Validation: 11,445


In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import time

# Prepare validation data
X_pre_train = X_train.iloc[preselection_train_idx]
X_pre_val = X_train.iloc[preselection_val_idx]

y_pre_train = y_train[preselection_train_idx]
y_pre_val = y_train[preselection_val_idx]

rf_preselection_results = []

# Evaluate the four configurations
for i, config in enumerate(rf_configs, start=1):

    model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            **config
        ))
    ])

    start_time = time.time()

    model.fit(X_pre_train, y_pre_train)

    y_pred = model.predict(X_pre_val)
    y_prob = model.predict_proba(X_pre_val)

    rf_preselection_results.append({
        "Configuration": i,
        **config,
        "Macro-F1": f1_score(y_pre_val, y_pred, average="macro"),
        "Log Loss": log_loss(
            y_pre_val,
            y_prob,
            labels=np.arange(len(target_mapping))
        ),
        "Time (min)": (time.time() - start_time) / 60
    })

    print(f"Configuration {i} completed.")

rf_preselection_df = pd.DataFrame(rf_preselection_results)

display(rf_preselection_df)

Configuration 1 completed.
Configuration 2 completed.
Configuration 3 completed.
Configuration 4 completed.


,Configuration,max_depth,min_samples_leaf,max_features,Macro-F1,Log Loss,Time (min)
0,1,15,5,sqrt,0.336292,1.604209,0.467285
1,2,15,10,sqrt,0.331038,1.632820,0.485728
2,3,25,5,sqrt,0.342643,1.549094,0.690249
3,4,25,10,sqrt,0.333948,1.601401,0.546898


In [26]:
# Select the best configuration based on Macro-F1
best_row = rf_preselection_df.loc[
    rf_preselection_df["Macro-F1"].idxmax()
]

rf_best_config = {
    "max_depth": int(best_row["max_depth"]),
    "min_samples_leaf": int(best_row["min_samples_leaf"]),
    "max_features": best_row["max_features"]
}

print("Best configuration:", rf_best_config)

# Build optimized Random Forest pipeline
rf_optimized = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        **rf_best_config
    ))
])

print("Optimized Random Forest successfully defined.")

Best configuration: {'max_depth': 25, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
Optimized Random Forest successfully defined.


In [27]:
# Evaluate optimized Random Forest using the original 5 folds
rf_optimized_cv = cross_validate(
    rf_optimized,
    X_train,
    y_train,
    cv=cv_folds,
    scoring={
        "macro_f1": "f1_macro",
        "log_loss": "neg_log_loss"
    },
    n_jobs=1,
    return_train_score=False
)

# Summarize fold results
rf_optimized_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro-F1": rf_optimized_cv["test_macro_f1"],
    "Log Loss": -rf_optimized_cv["test_log_loss"]
})

display(rf_optimized_summary)

print("\nMean results:")
print(rf_optimized_summary[["Macro-F1", "Log Loss"]].mean())

print("\nStandard deviation:")
print(rf_optimized_summary[["Macro-F1", "Log Loss"]].std())

,Fold,Macro-F1,Log Loss
0,1,0.326832,1.574805
1,2,0.335722,1.555011
2,3,0.337700,1.556519
3,4,0.330825,1.566580
4,5,0.338178,1.556609



Mean results:
Macro-F1    0.333852
Log Loss    1.561905
dtype: float64

Standard deviation:
Macro-F1    0.004885
Log Loss    0.008556
dtype: float64


### 3.2 Entrenamiento Mejor Modelo

In [28]:
# Fit the final optimized Random Forest on the complete training set
rf_final = clone(rf_optimized)

rf_final.fit(X_train, y_train)

print("Final Random Forest successfully trained.")

Final Random Forest successfully trained.


In [29]:
# Extract feature importances from the optimized Random Forest
classifier = rf_final.named_steps["classifier"]
fitted_preprocessor = rf_final.named_steps["preprocessor"]

importances = classifier.feature_importances_

# Initialize importance dictionary
feature_importance = {}

# Binary and numerical features
for feature in binary_features + numerical_features:
    transformer = "binary" if feature in binary_features else "numerical"
    feature_list = binary_features if feature in binary_features else numerical_features

    position = (
        fitted_preprocessor.output_indices_[transformer].start
        + feature_list.index(feature)
    )

    feature_importance[feature] = importances[position]

# Aggregate one-hot encoded categorical features
encoder = fitted_preprocessor.named_transformers_["categorical"]
categorical_names = encoder.get_feature_names_out(categorical_features)
categorical_start = fitted_preprocessor.output_indices_["categorical"].start

for feature in categorical_features:
    positions = [
        categorical_start + i
        for i, name in enumerate(categorical_names)
        if name.startswith(feature + "_")
    ]

    feature_importance[feature] = importances[positions].sum()

# Create and display importance table
rf_optimized_importance = (
    pd.DataFrame({
        "Variable": feature_importance.keys(),
        "Importance": feature_importance.values()
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(rf_optimized_importance)

,Variable,Importance
0,edad,0.262583
1,zat_centroid_distance_m,0.255654
2,has_motorcycle_license,0.102315
3,estra_hg,0.099006
4,time_period_6,0.071630
5,vehicle_availability,0.065748
6,purpose_group,0.058807
7,has_car_license,0.051653
8,sexo,0.032604


In [30]:
# Evaluate the optimized Random Forest on the test set
from sklearn.metrics import f1_score, log_loss, accuracy_score

y_test_pred = rf_final.predict(X_test)
y_test_proba = rf_final.predict_proba(X_test)

rf_test_results = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Macro-F1": f1_score(y_test, y_test_pred, average="macro"),
    "Log Loss": log_loss(
        y_test,
        y_test_proba,
        labels=np.arange(len(target_classes))
    )
}

display(pd.DataFrame([rf_test_results], index=["Random Forest Optimized"]).T)

,Random Forest Optimized
Accuracy,0.445654
Macro-F1,0.341819
Log Loss,1.539070


### 3.3 Guardar Modelo Seleccionado

In [32]:
# Save the final optimized Random Forest artifacts
import joblib
import json

# Save the complete fitted pipeline
joblib.dump(rf_final, ml_path / "RF_final_model.joblib")

# Save model configuration and evaluation results
rf_metadata = {
    "model": "Random Forest",
    "status": "Final",
    "features": b3_features,
    "target": "modo_principal_agrupado",
    "target_classes": target_classes,
    "hyperparameters": rf_best_config,
    "n_estimators": 300,
    "class_weight": "balanced",
    "random_state": 42,
    "cv_macro_f1": 0.3339,
    "cv_log_loss": 1.5619,
    "test_metrics": {
        key: float(value)
        for key, value in rf_test_results.items()
    }
}

with open(ml_path / "RF_final_metadata.json", "w") as file:
    json.dump(rf_metadata, file, indent=4)

# Save feature importance
rf_optimized_importance.to_csv(
    ml_path / "RF_final_feature_importance.csv",
    index=False
)

print("Random Forest artifacts saved successfully.")
print("Saved files:")
for file in sorted(ml_path.glob("RF_final_*")):
    print("-", file.name)

Random Forest artifacts saved successfully.
Saved files:
- RF_final_feature_importance.csv
- RF_final_metadata.json
- RF_final_model.joblib


## 3 XGBoost

In [36]:
import xgboost
import torch
import shap
import optuna
import sklearn
import numpy
import pandas

print("XGBoost:", xgboost.__version__)
print("PyTorch:", torch.__version__)
print("SHAP:", shap.__version__)
print("Optuna:", optuna.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XGBoost: 3.4.1
PyTorch: 2.15.0+cpu
SHAP: 0.54.1
Optuna: 5.1.0
Scikit-learn: 1.9.1
CUDA available: False


### 3.1 evaluacion Inicial

In [37]:
# XGBoost baseline: preprocessing and model definition
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# Tree-based preprocessing: no numerical scaling
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("binary", "passthrough", binary_features),
        ("categorical", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ), categorical_features),
        ("numerical", "passthrough", numerical_features)
    ],
    remainder="drop"
)

# Define baseline XGBoost
xgb_baseline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(target_classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

print("XGBoost version:", xgb.__version__)
print("Baseline pipeline successfully defined.")

XGBoost version: 3.4.1
Baseline pipeline successfully defined.


In [38]:
# Evaluate XGBoost baseline using the original 5 CV folds
from sklearn.base import clone
from sklearn.metrics import f1_score, log_loss
from sklearn.utils.class_weight import compute_sample_weight
import time

xgb_baseline_results = []

for fold_number, (train_indices, val_indices) in enumerate(cv_folds, start=1):

    start_time = time.time()

    X_fold_train = X_train.iloc[train_indices]
    X_fold_val = X_train.iloc[val_indices]

    y_fold_train = y_train[train_indices]
    y_fold_val = y_train[val_indices]

    # Calculate balanced sample weights using training data only
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_fold_train
    )

    # Train model
    fold_model = clone(xgb_baseline)

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        classifier__sample_weight=sample_weights
    )

    # Predictions
    y_pred = fold_model.predict(X_fold_val)
    y_proba = fold_model.predict_proba(X_fold_val)

    # Evaluation
    fold_f1 = f1_score(y_fold_val, y_pred, average="macro")
    fold_loss = log_loss(
        y_fold_val,
        y_proba,
        labels=np.arange(len(target_classes))
    )

    elapsed_time = (time.time() - start_time) / 60

    xgb_baseline_results.append({
        "Fold": fold_number,
        "Macro-F1": fold_f1,
        "Log Loss": fold_loss,
        "Time (min)": elapsed_time
    })

    print(
        f"Fold {fold_number} | "
        f"Macro-F1: {fold_f1:.4f} | "
        f"Log Loss: {fold_loss:.4f} | "
        f"Time: {elapsed_time:.2f} min"
    )

xgb_baseline_df = pd.DataFrame(xgb_baseline_results)

print("\nBaseline CV Summary:")
display(
    xgb_baseline_df[
        ["Macro-F1", "Log Loss", "Time (min)"]
    ].agg(["mean", "std"])
)

Fold 1 | Macro-F1: 0.3422 | Log Loss: 1.5465 | Time: 0.55 min
Fold 2 | Macro-F1: 0.3525 | Log Loss: 1.5207 | Time: 0.44 min
Fold 3 | Macro-F1: 0.3536 | Log Loss: 1.5206 | Time: 0.44 min
Fold 4 | Macro-F1: 0.3506 | Log Loss: 1.5354 | Time: 0.44 min
Fold 5 | Macro-F1: 0.3458 | Log Loss: 1.5350 | Time: 0.43 min

Baseline CV Summary:


,Macro-F1,Log Loss,Time (min)
mean,0.348940,1.531652,0.462757
std,0.004802,0.011071,0.050988


In [39]:
# Fit XGBoost baseline on the complete training set for feature importance
xgb_fi_model = clone(xgb_baseline)

xgb_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_fi_model.fit(
    X_train,
    y_train,
    classifier__sample_weight=xgb_sample_weights
)

# Extract feature importances
classifier = xgb_fi_model.named_steps["classifier"]
fitted_preprocessor = xgb_fi_model.named_steps["preprocessor"]

importances = classifier.feature_importances_
feature_names = fitted_preprocessor.get_feature_names_out()

# Aggregate importance by original variable
xgb_feature_importance = {}

for feature in binary_features + numerical_features:
    positions = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1] == feature
    ]
    xgb_feature_importance[feature] = importances[positions].sum()

for feature in categorical_features:
    positions = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1].startswith(feature + "_")
    ]
    xgb_feature_importance[feature] = importances[positions].sum()

xgb_baseline_importance = (
    pd.DataFrame({
        "Variable": xgb_feature_importance.keys(),
        "Importance": xgb_feature_importance.values()
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(xgb_baseline_importance)

,Variable,Importance
0,has_motorcycle_license,0.207924
1,estra_hg,0.199251
2,purpose_group,0.178612
3,time_period_6,0.100976
4,has_car_license,0.093209
5,vehicle_availability,0.089733
6,sexo,0.047013
7,edad,0.043949
8,zat_centroid_distance_m,0.039332


### 3.2 Optimizacion XGBoost

In [40]:
# XGBoost hyperparameter preselection
from sklearn.metrics import f1_score, log_loss
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
import time

xgb_configs = [
    {"max_depth": depth, "min_child_weight": child_weight}
    for depth in [4, 6, 8]
    for child_weight in [1, 5]
]

X_pre_train = X_train.iloc[preselection_train_idx]
X_pre_val = X_train.iloc[preselection_val_idx]

y_pre_train = y_train[preselection_train_idx]
y_pre_val = y_train[preselection_val_idx]

preselection_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_pre_train
)

xgb_preselection_results = []

for config_number, config in enumerate(xgb_configs, start=1):

    start_time = time.time()

    model = Pipeline([
        ("preprocessor", clone(xgb_preprocessor)),
        ("classifier", xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=len(target_classes),
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            **config
        ))
    ])

    model.fit(
        X_pre_train,
        y_pre_train,
        classifier__sample_weight=preselection_weights
    )

    y_pred = model.predict(X_pre_val)
    y_proba = model.predict_proba(X_pre_val)

    xgb_preselection_results.append({
        **config,
        "Macro-F1": f1_score(y_pre_val, y_pred, average="macro"),
        "Log Loss": log_loss(
            y_pre_val,
            y_proba,
            labels=np.arange(len(target_classes))
        ),
        "Time (min)": (time.time() - start_time) / 60
    })

    print(
        f"Configuration {config_number}/6 | "
        f"Macro-F1: {xgb_preselection_results[-1]['Macro-F1']:.4f} | "
        f"Log Loss: {xgb_preselection_results[-1]['Log Loss']:.4f}"
    )

xgb_preselection_df = (
    pd.DataFrame(xgb_preselection_results)
    .sort_values(
        ["Macro-F1", "Log Loss"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(xgb_preselection_df)

Configuration 1/6 | Macro-F1: 0.3414 | Log Loss: 1.5955
Configuration 2/6 | Macro-F1: 0.3409 | Log Loss: 1.5966
Configuration 3/6 | Macro-F1: 0.3494 | Log Loss: 1.5241
Configuration 4/6 | Macro-F1: 0.3472 | Log Loss: 1.5314
Configuration 5/6 | Macro-F1: 0.3582 | Log Loss: 1.4469
Configuration 6/6 | Macro-F1: 0.3536 | Log Loss: 1.4650


,max_depth,min_child_weight,Macro-F1,Log Loss,Time (min)
0,8,1,0.358211,1.446850,0.614574
1,8,5,0.353579,1.465013,0.553130
2,6,1,0.349378,1.524073,0.436631
3,6,5,0.347227,1.531389,0.431249
4,4,1,0.341409,1.595500,0.442826
5,4,5,0.340909,1.596555,0.369398


In [41]:
# Extended XGBoost hyperparameter preselection
xgb_extended_configs = [
    {"max_depth": depth, "min_child_weight": child_weight}
    for depth in [10, 12]
    for child_weight in [1, 3, 5]
]

xgb_extended_results = []

for config_number, config in enumerate(xgb_extended_configs, start=1):

    start_time = time.time()

    model = Pipeline([
        ("preprocessor", clone(xgb_preprocessor)),
        ("classifier", xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=len(target_classes),
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            **config
        ))
    ])

    model.fit(
        X_pre_train,
        y_pre_train,
        classifier__sample_weight=preselection_weights
    )

    y_pred = model.predict(X_pre_val)
    y_proba = model.predict_proba(X_pre_val)

    xgb_extended_results.append({
        **config,
        "Macro-F1": f1_score(y_pre_val, y_pred, average="macro"),
        "Log Loss": log_loss(
            y_pre_val,
            y_proba,
            labels=np.arange(len(target_classes))
        ),
        "Time (min)": (time.time() - start_time) / 60
    })

    print(
        f"Configuration {config_number}/6 | "
        f"Depth: {config['max_depth']} | "
        f"Child weight: {config['min_child_weight']} | "
        f"Macro-F1: {xgb_extended_results[-1]['Macro-F1']:.4f} | "
        f"Log Loss: {xgb_extended_results[-1]['Log Loss']:.4f}"
    )

# Combine original and extended results
xgb_all_preselection_df = pd.concat(
    [
        xgb_preselection_df,
        pd.DataFrame(xgb_extended_results)
    ],
    ignore_index=True
).sort_values(
    ["Macro-F1", "Log Loss"],
    ascending=[False, True]
).reset_index(drop=True)

display(xgb_all_preselection_df)

Configuration 1/6 | Depth: 10 | Child weight: 1 | Macro-F1: 0.3620 | Log Loss: 1.3889
Configuration 2/6 | Depth: 10 | Child weight: 3 | Macro-F1: 0.3598 | Log Loss: 1.4041
Configuration 3/6 | Depth: 10 | Child weight: 5 | Macro-F1: 0.3616 | Log Loss: 1.4171
Configuration 4/6 | Depth: 12 | Child weight: 1 | Macro-F1: 0.3613 | Log Loss: 1.3644
Configuration 5/6 | Depth: 12 | Child weight: 3 | Macro-F1: 0.3552 | Log Loss: 1.3791
Configuration 6/6 | Depth: 12 | Child weight: 5 | Macro-F1: 0.3577 | Log Loss: 1.3894


,max_depth,min_child_weight,Macro-F1,Log Loss,Time (min)
0,10,1,0.362005,1.388876,0.724573
1,10,5,0.361620,1.417090,0.657552
2,12,1,0.361304,1.364386,0.940707
3,10,3,0.359845,1.404132,0.688254
4,8,1,0.358211,1.446850,0.614574
5,12,5,0.357744,1.389385,0.779892
6,12,3,0.355177,1.379072,0.815001
7,8,5,0.353579,1.465013,0.553130
8,6,1,0.349378,1.524073,0.436631
9,6,5,0.347227,1.531389,0.431249


### 3.3 Validación cruzada del XGBoost optimizado

In [42]:
# Define optimized XGBoost model
xgb_best_config = {
    "max_depth": 10,
    "min_child_weight": 1
}

xgb_optimized = Pipeline([
    ("preprocessor", clone(xgb_preprocessor)),
    ("classifier", xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(target_classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **xgb_best_config
    ))
])

# Evaluate optimized XGBoost using the original five folds
xgb_optimized_results = []

for fold_number, (train_indices, val_indices) in enumerate(cv_folds, start=1):

    start_time = time.time()

    X_fold_train = X_train.iloc[train_indices]
    X_fold_val = X_train.iloc[val_indices]

    y_fold_train = y_train[train_indices]
    y_fold_val = y_train[val_indices]

    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_fold_train
    )

    fold_model = clone(xgb_optimized)

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        classifier__sample_weight=sample_weights
    )

    y_pred = fold_model.predict(X_fold_val)
    y_proba = fold_model.predict_proba(X_fold_val)

    xgb_optimized_results.append({
        "Fold": fold_number,
        "Macro-F1": f1_score(y_fold_val, y_pred, average="macro"),
        "Log Loss": log_loss(
            y_fold_val,
            y_proba,
            labels=np.arange(len(target_classes))
        ),
        "Time (min)": (time.time() - start_time) / 60
    })

    print(
        f"Fold {fold_number} | "
        f"Macro-F1: {xgb_optimized_results[-1]['Macro-F1']:.4f} | "
        f"Log Loss: {xgb_optimized_results[-1]['Log Loss']:.4f}"
    )

xgb_optimized_cv_df = pd.DataFrame(xgb_optimized_results)

print("\nOptimized XGBoost CV Summary:")
display(
    xgb_optimized_cv_df[
        ["Macro-F1", "Log Loss", "Time (min)"]
    ].agg(["mean", "std"])
)

Fold 1 | Macro-F1: 0.3542 | Log Loss: 1.4185
Fold 2 | Macro-F1: 0.3656 | Log Loss: 1.3897
Fold 3 | Macro-F1: 0.3610 | Log Loss: 1.3927
Fold 4 | Macro-F1: 0.3582 | Log Loss: 1.4046
Fold 5 | Macro-F1: 0.3574 | Log Loss: 1.3993

Optimized XGBoost CV Summary:


,Macro-F1,Log Loss,Time (min)
mean,0.359284,1.400981,0.714508
std,0.004275,0.011358,0.041299


In [43]:
# Fit the final optimized XGBoost on the complete training set
xgb_final = clone(xgb_optimized)

xgb_final_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_final.fit(
    X_train,
    y_train,
    classifier__sample_weight=xgb_final_weights
)

print("Final XGBoost successfully trained.")

Final XGBoost successfully trained.


### 3.4 Importancia de variables XGBoost

In [44]:
# Extract feature importance from the final XGBoost model
classifier = xgb_final.named_steps["classifier"]
fitted_preprocessor = xgb_final.named_steps["preprocessor"]

importances = classifier.feature_importances_
feature_names = fitted_preprocessor.get_feature_names_out()

# Aggregate importance by original variable
xgb_feature_importance = {}

for feature in binary_features + numerical_features:
    positions = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1] == feature
    ]
    xgb_feature_importance[feature] = importances[positions].sum()

for feature in categorical_features:
    positions = [
        i for i, name in enumerate(feature_names)
        if name.split("__")[-1].startswith(feature + "_")
    ]
    xgb_feature_importance[feature] = importances[positions].sum()

xgb_final_importance = (
    pd.DataFrame({
        "Variable": xgb_feature_importance.keys(),
        "Importance": xgb_feature_importance.values()
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(xgb_final_importance)

,Variable,Importance
0,estra_hg,0.218800
1,purpose_group,0.204132
2,has_motorcycle_license,0.174104
3,time_period_6,0.115065
4,vehicle_availability,0.085655
5,has_car_license,0.073884
6,sexo,0.046854
7,zat_centroid_distance_m,0.041115
8,edad,0.040391


### 3.5 Evaluacion En test XGBoost

In [45]:
# Evaluate final XGBoost on the untouched test set
from sklearn.metrics import accuracy_score, f1_score, log_loss

xgb_test_pred = xgb_final.predict(X_test)
xgb_test_proba = xgb_final.predict_proba(X_test)

xgb_test_results = {
    "Accuracy": accuracy_score(y_test, xgb_test_pred),
    "Macro-F1": f1_score(y_test, xgb_test_pred, average="macro"),
    "Log Loss": log_loss(
        y_test,
        xgb_test_proba,
        labels=np.arange(len(target_classes))
    )
}

display(
    pd.DataFrame(
        [xgb_test_results],
        index=["XGBoost Optimized"]
    ).T
)

,XGBoost Optimized
Accuracy,0.502421
Macro-F1,0.379608
Log Loss,1.384822


### 3.6 Guardat Modelo XGBOOST

In [46]:
# Save final XGBoost model and evaluation artifacts
import joblib
import json

joblib.dump(
    xgb_final,
    ml_path / "XGB_final_model.joblib"
)

xgb_metadata = {
    "model": "XGBoost",
    "status": "Final",
    "features": b3_features,
    "target": "modo_principal_agrupado",
    "target_classes": target_classes,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 10,
        "min_child_weight": 1,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "multi:softprob",
        "class_weight": "balanced",
        "random_state": 42
    },
    "cv_metrics": {
        "macro_f1_mean": 0.359284,
        "macro_f1_std": 0.004275,
        "log_loss_mean": 1.400981,
        "log_loss_std": 0.011358
    },
    "test_metrics": {
        key: float(value)
        for key, value in xgb_test_results.items()
    }
}

with open(ml_path / "XGB_final_metadata.json", "w") as f:
    json.dump(xgb_metadata, f, indent=4)

xgb_final_importance.to_csv(
    ml_path / "XGB_final_feature_importance.csv",
    index=False
)

print("XGBoost artifacts saved successfully.")

XGBoost artifacts saved successfully.


## 4. ANN Artificial Neuronal Network

In [47]:
# Check PyTorch environment and computational device
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.15.0+cpu
CUDA available: False
Selected device: cpu


In [50]:
# Configure preprocessing for the neural network
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nn_preprocessor = ColumnTransformer(
    transformers=[
        ("binary", "passthrough", binary_features),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        ("numerical", StandardScaler(), numerical_features)
    ],
    remainder="drop"
)

print("Neural network preprocessor configured.")
print("Binary features:", binary_features)
print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Neural network preprocessor configured.
Binary features: ['has_car_license', 'has_motorcycle_license']
Categorical features: ['sexo', 'estra_hg', 'vehicle_availability', 'purpose_group', 'time_period_6']
Numerical features: ['edad', 'zat_centroid_distance_m']


In [51]:
# Inspect the transformed feature matrix
nn_preprocessor_check = nn_preprocessor.fit_transform(X_train)

print("Original training shape:", X_train.shape)
print("Transformed training shape:", nn_preprocessor_check.shape)
print("Data type:", nn_preprocessor_check.dtype)
print("Contains missing values:", np.isnan(nn_preprocessor_check).any())

Original training shape: (80114, 9)
Transformed training shape: (80114, 26)
Data type: float64
Contains missing values: False


### 3.1 Definicion de la red Base

In [52]:
# Define the three-hidden-layer MLP architecture
import torch.nn as nn

class ModeChoiceMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)


# Initialize model for inspection
nn_model_check = ModeChoiceMLP(
    input_dim=26,
    num_classes=len(target_classes)
)

print(nn_model_check)

ModeChoiceMLP(
  (network): Sequential(
    (0): Linear(in_features=26, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=64, out_features=11, bias=True)
  )
)


In [53]:
# Analyze class distribution in the training dataset
class_counts = pd.Series(y_train).value_counts().sort_index()

class_distribution = pd.DataFrame({
    "Class": target_classes,
    "Observations": class_counts.values,
    "Percentage": (
        class_counts.values / len(y_train) * 100
    )
})

display(
    class_distribution.sort_values(
        "Observations",
        ascending=False
    ).reset_index(drop=True)
)

,Class,Observations,Percentage
0,TRANSPORTE PÚBLICO,23520,29.358165
1,A PIE > 15 MIN,19939,24.888284
2,AUTO,10066,12.564595
3,A PIE <15 MIN,9909,12.368625
4,MOTO,4998,6.238610
5,BICICLETA,4881,6.092568
6,TAXI OCUPADO,2612,3.260354
7,TRANSPORTE ESCOLAR,1646,2.054572
8,ESPECIAL OCUPADO,1585,1.978431
9,INFORMAL,693,0.865017


In [54]:
# Define a function to calculate balanced class weights
def calculate_class_weights(y, num_classes):
    class_counts = torch.bincount(
        torch.as_tensor(y, dtype=torch.long),
        minlength=num_classes
    ).float()

    total_samples = class_counts.sum()

    class_weights = total_samples / (
        num_classes * class_counts.clamp(min=1)
    )

    return class_weights


# Inspect class weights using the full training partition
nn_class_weights_check = calculate_class_weights(
    y_train,
    len(target_classes)
)

class_weights_table = pd.DataFrame({
    "Class": target_classes,
    "Weight": nn_class_weights_check.numpy(),
    "Observations": np.bincount(
        y_train,
        minlength=len(target_classes)
    )
})

display(
    class_weights_table.sort_values(
        "Weight",
        ascending=False
    ).reset_index(drop=True)
)

,Class,Weight,Observations
0,OTRO,27.483362,265
1,INFORMAL,10.509511,693
2,ESPECIAL OCUPADO,4.595010,1585
3,TRANSPORTE ESCOLAR,4.424721,1646
4,TAXI OCUPADO,2.788320,2612
5,BICICLETA,1.492131,4881
6,MOTO,1.457201,4998
7,A PIE <15 MIN,0.734998,9909
8,AUTO,0.723534,10066
9,A PIE > 15 MIN,0.365269,19939


### 4.2 Entrenamiento de la RED

In [55]:
# Define the neural network training function
from torch.utils.data import TensorDataset, DataLoader

def train_mlp_epoch(model, loader, optimizer, criterion, device):

    model.train()
    total_loss = 0.0
    total_samples = 0

    for batch_X, batch_y in loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        batch_size = batch_y.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate_mlp_loss(model, loader, criterion, device):

    model.eval()
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch_X, batch_y in loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            loss = criterion(logits, batch_y)

            batch_size = batch_y.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples


print("MLP training and evaluation functions configured.")

MLP training and evaluation functions configured.


In [56]:
# Define MLP training with early stopping
import copy

def fit_mlp(
    model,
    train_loader,
    val_loader,
    class_weights,
    learning_rate=0.001,
    weight_decay=0.0001,
    max_epochs=100,
    patience=10,
    device=device
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device)
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "val_loss": []
    }

    for epoch in range(max_epochs):

        train_loss = train_mlp_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss = evaluate_mlp_loss(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1:03d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Validation Loss: {val_loss:.4f}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}.")
            break

    model.load_state_dict(best_state)

    print(f"Best epoch: {best_epoch}")
    print(f"Best validation loss: {best_val_loss:.4f}")

    return model, history, best_epoch

In [58]:
# Prepare training and validation data for the MLP experiment
from sklearn.base import clone
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Split the preselection partition
X_nn_train = X_train.iloc[preselection_train_idx]
X_nn_val = X_train.iloc[preselection_val_idx]

y_nn_train = y_train[preselection_train_idx]
y_nn_val = y_train[preselection_val_idx]

# Fit preprocessing only on training data
nn_preprocessor_exp = clone(nn_preprocessor)

X_nn_train_transformed = nn_preprocessor_exp.fit_transform(X_nn_train)
X_nn_val_transformed = nn_preprocessor_exp.transform(X_nn_val)

# Convert features and targets to PyTorch tensors
X_nn_train_tensor = torch.tensor(
    X_nn_train_transformed, dtype=torch.float32
)
X_nn_val_tensor = torch.tensor(
    X_nn_val_transformed, dtype=torch.float32
)

y_nn_train_tensor = torch.tensor(y_nn_train, dtype=torch.long)
y_nn_val_tensor = torch.tensor(y_nn_val, dtype=torch.long)

# Create DataLoaders
nn_train_dataset = TensorDataset(
    X_nn_train_tensor, y_nn_train_tensor
)
nn_val_dataset = TensorDataset(
    X_nn_val_tensor, y_nn_val_tensor
)

nn_train_loader = DataLoader(
    nn_train_dataset,
    batch_size=512,
    shuffle=True
)

nn_val_loader = DataLoader(
    nn_val_dataset,
    batch_size=512,
    shuffle=False
)

# Calculate class weights using training data only
nn_class_weights = calculate_class_weights(
    y_nn_train,
    len(target_classes)
)

print("Training observations:", len(nn_train_dataset))
print("Validation observations:", len(nn_val_dataset))
print("Input features:", X_nn_train_tensor.shape[1])
print("Number of classes:", len(target_classes))
print("Class weights shape:", nn_class_weights.shape)

Training observations: 68669
Validation observations: 11445
Input features: 26
Number of classes: 11
Class weights shape: torch.Size([11])


### 4.3 Entrenar el modelo Baseline

In [59]:
# Train the baseline MLP model
torch.manual_seed(42)

nn_model_baseline = ModeChoiceMLP(
    input_dim=X_nn_train_tensor.shape[1],
    num_classes=len(target_classes)
)

nn_model_baseline, nn_history, nn_best_epoch = fit_mlp(
    model=nn_model_baseline,
    train_loader=nn_train_loader,
    val_loader=nn_val_loader,
    class_weights=nn_class_weights,
    learning_rate=0.001,
    weight_decay=0.0001,
    max_epochs=100,
    patience=10,
    device=device
)

Epoch 010 | Train Loss: 1.7373 | Validation Loss: 1.6642
Epoch 020 | Train Loss: 1.6771 | Validation Loss: 1.6444
Epoch 030 | Train Loss: 1.6344 | Validation Loss: 1.6499
Early stopping at epoch 31.
Best epoch: 21
Best validation loss: 1.6389


### 4.4 Evaluacion del Nivel Base

In [60]:
# Evaluate the baseline MLP on the validation set
from sklearn.metrics import accuracy_score, f1_score, log_loss

nn_model_baseline.eval()

with torch.no_grad():
    nn_val_logits = nn_model_baseline(
        X_nn_val_tensor.to(device)
    )
    nn_val_proba = torch.softmax(
        nn_val_logits, dim=1
    ).cpu().numpy()

nn_val_pred = nn_val_proba.argmax(axis=1)

nn_baseline_results = {
    "Accuracy": accuracy_score(y_nn_val, nn_val_pred),
    "Macro-F1": f1_score(
        y_nn_val, nn_val_pred, average="macro"
    ),
    "Log Loss": log_loss(
        y_nn_val,
        nn_val_proba,
        labels=np.arange(len(target_classes))
    )
}

display(
    pd.DataFrame(
        [nn_baseline_results],
        index=["MLP Baseline"]
    ).T
)

,MLP Baseline
Accuracy,0.396680
Macro-F1,0.316532
Log Loss,1.653530


In [61]:
# Evaluate MLP performance by transport mode
from sklearn.metrics import classification_report

nn_classification_report = classification_report(
    y_nn_val,
    nn_val_pred,
    labels=np.arange(len(target_classes)),
    target_names=target_classes,
    output_dict=True,
    zero_division=0
)

nn_class_metrics = pd.DataFrame(
    nn_classification_report
).T

display(
    nn_class_metrics.loc[
        target_classes,
        ["precision", "recall", "f1-score", "support"]
    ].sort_values("support", ascending=False)
)

,precision,recall,f1-score,support
TRANSPORTE PÚBLICO,0.782072,0.433631,0.557917,3360.0
A PIE > 15 MIN,0.566540,0.104635,0.176645,2848.0
AUTO,0.540618,0.522949,0.531637,1438.0
A PIE <15 MIN,0.359404,0.545583,0.433343,1415.0
MOTO,0.573146,0.801120,0.668224,714.0
BICICLETA,0.338346,0.387374,0.361204,697.0
TAXI OCUPADO,0.175174,0.404826,0.244534,373.0
TRANSPORTE ESCOLAR,0.173532,0.838983,0.287582,236.0
ESPECIAL OCUPADO,0.114695,0.140969,0.126482,227.0
INFORMAL,0.032383,0.252525,0.057405,99.0


In [62]:
# Analyze the MLP confusion matrix
from sklearn.metrics import confusion_matrix

nn_confusion = confusion_matrix(
    y_nn_val,
    nn_val_pred,
    labels=np.arange(len(target_classes))
)

nn_confusion_df = pd.DataFrame(
    nn_confusion,
    index=target_classes,
    columns=target_classes
)

display(nn_confusion_df)

,AUTO,A PIE <15 MIN,A PIE > 15 MIN,BICICLETA,ESPECIAL OCUPADO,INFORMAL,MOTO,OTRO,TAXI OCUPADO,TRANSPORTE ESCOLAR,TRANSPORTE PÚBLICO
AUTO,752,53,9,37,16,21,129,200,121,54,46
A PIE <15 MIN,106,772,106,67,10,72,32,27,20,147,56
A PIE > 15 MIN,148,1094,298,164,48,237,63,70,158,434,134
BICICLETA,78,47,12,270,15,19,67,78,16,42,53
ESPECIAL OCUPADO,30,14,4,18,32,23,15,29,23,20,19
INFORMAL,0,9,4,6,7,25,7,3,7,9,22
MOTO,14,11,4,25,5,6,572,31,13,7,26
OTRO,2,1,1,2,0,1,10,13,3,1,4
TAXI OCUPADO,53,17,19,21,2,29,9,16,151,13,43
TRANSPORTE ESCOLAR,3,20,2,1,1,5,0,2,1,198,3


### 4.5 Optimizacion de la red

In [63]:
# Generate class weights with different balancing intensities

class_counts = np.bincount(y_nn_train, minlength=len(target_classes))
base_weights = len(y_nn_train) / (len(target_classes) * class_counts)

weight_exponents = [0.00, 0.25, 0.50, 0.75]

nn_weight_experiments = {}

for alpha in weight_exponents:
    weights = base_weights ** alpha
    weights = weights / weights.mean()

    nn_weight_experiments[alpha] = torch.tensor(
        weights, dtype=torch.float32
    )

print("Prepared class-weight experiments:")

for alpha, weights in nn_weight_experiments.items():
    print(f"Alpha = {alpha:.2f} | "
          f"Min weight = {weights.min():.3f} | "
          f"Max weight = {weights.max():.3f}")

Prepared class-weight experiments:
Alpha = 0.00 | Min weight = 1.000 | Max weight = 1.000
Alpha = 0.25 | Min weight = 0.591 | Max weight = 1.816
Alpha = 0.50 | Min weight = 0.311 | Max weight = 2.928
Alpha = 0.75 | Min weight = 0.146 | Max weight = 4.216


In [64]:
# Train MLP models with different class-weight intensities

nn_weight_results = []

for alpha, class_weights in nn_weight_experiments.items():

    print(f"\nTraining MLP with alpha = {alpha:.2f}")

    torch.manual_seed(42)
    np.random.seed(42)

    model = ModeChoiceMLP(
        input_dim=X_nn_train_tensor.shape[1],
        num_classes=len(target_classes)
    )

    model, history, best_epoch = fit_mlp(
        model,
        nn_train_loader,
        nn_val_loader,
        class_weights,
        learning_rate=0.001,
        weight_decay=0.0001,
        max_epochs=100,
        patience=10,
        device=device
    )

    model.eval()

    with torch.no_grad():
        logits = model(X_nn_val_tensor.to(device))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

    predictions = probabilities.argmax(axis=1)

    nn_weight_results.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_nn_val, predictions),
        "Macro-F1": f1_score(y_nn_val, predictions, average="macro"),
        "Log Loss": log_loss(
            y_nn_val,
            probabilities,
            labels=np.arange(len(target_classes))
        ),
        "Best Epoch": best_epoch,
        "Model": model,
        "History": history,
        "Predictions": predictions,
        "Probabilities": probabilities
    })

nn_weight_comparison = pd.DataFrame([
    {k: v for k, v in result.items()
     if k not in ["Model", "History", "Predictions", "Probabilities"]}
    for result in nn_weight_results
])

display(nn_weight_comparison.round(4))


Training MLP with alpha = 0.00
Epoch 010 | Train Loss: 1.3064 | Validation Loss: 1.2566
Epoch 020 | Train Loss: 1.2779 | Validation Loss: 1.2378
Epoch 030 | Train Loss: 1.2620 | Validation Loss: 1.2300
Epoch 040 | Train Loss: 1.2543 | Validation Loss: 1.2259
Epoch 050 | Train Loss: 1.2498 | Validation Loss: 1.2266
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.2252

Training MLP with alpha = 0.25
Epoch 010 | Train Loss: 1.4371 | Validation Loss: 1.3831
Epoch 020 | Train Loss: 1.4052 | Validation Loss: 1.3608
Epoch 030 | Train Loss: 1.3880 | Validation Loss: 1.3563
Epoch 040 | Train Loss: 1.3796 | Validation Loss: 1.3508
Epoch 050 | Train Loss: 1.3726 | Validation Loss: 1.3504
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.3479

Training MLP with alpha = 0.50
Epoch 010 | Train Loss: 1.5675 | Validation Loss: 1.5080
Epoch 020 | Train Loss: 1.5325 | Validation Loss: 1.4866
Epoch 030 | Train Loss: 1.5106 | Validation Loss: 1.4783
Epoch 040 | Train L

,Alpha,Accuracy,Macro-F1,Log Loss,Best Epoch
0,0.00,0.5685,0.3041,1.2252,45
1,0.25,0.5681,0.3298,1.2391,45
2,0.50,0.5575,0.3647,1.3033,45
3,0.75,0.4927,0.3659,1.4636,33


In [65]:
# Evaluate the MLP model with alpha = 0.50 by class

best_nn_experiment = next(
    result for result in nn_weight_results
    if result["Alpha"] == 0.50
)

nn_alpha50_pred = best_nn_experiment["Predictions"]

nn_alpha50_report = pd.DataFrame(
    classification_report(
        y_nn_val,
        nn_alpha50_pred,
        labels=np.arange(len(target_classes)),
        target_names=target_classes,
        output_dict=True,
        zero_division=0
    )
).T

display(
    nn_alpha50_report.loc[
        target_classes,
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

,precision,recall,f1-score,support
AUTO,0.531,0.695,0.602,1438.0
A PIE <15 MIN,0.467,0.210,0.290,1415.0
A PIE > 15 MIN,0.530,0.627,0.574,2848.0
BICICLETA,0.386,0.359,0.372,697.0
ESPECIAL OCUPADO,0.207,0.075,0.110,227.0
INFORMAL,0.000,0.000,0.000,99.0
MOTO,0.597,0.794,0.682,714.0
OTRO,0.250,0.026,0.048,38.0
TAXI OCUPADO,0.318,0.204,0.248,373.0
TRANSPORTE ESCOLAR,0.295,0.606,0.397,236.0


In [66]:
# Compare F1-score across class-weight experiments

nn_class_comparison = pd.DataFrame(index=target_classes)

for result in nn_weight_results:
    alpha = result["Alpha"]

    report = classification_report(
        y_nn_val,
        result["Predictions"],
        labels=np.arange(len(target_classes)),
        target_names=target_classes,
        output_dict=True,
        zero_division=0
    )

    nn_class_comparison[f"F1_alpha_{alpha:.2f}"] = [
        report[class_name]["f1-score"]
        for class_name in target_classes
    ]

display(nn_class_comparison.round(3))

,F1_alpha_0.00,F1_alpha_0.25,F1_alpha_0.50,F1_alpha_0.75
AUTO,0.602,0.597,0.602,0.578
A PIE <15 MIN,0.025,0.131,0.290,0.408
A PIE > 15 MIN,0.609,0.604,0.574,0.439
BICICLETA,0.294,0.341,0.372,0.373
ESPECIAL OCUPADO,0.000,0.000,0.110,0.141
INFORMAL,0.000,0.000,0.000,0.106
MOTO,0.690,0.687,0.682,0.674
OTRO,0.000,0.000,0.048,0.048
TAXI OCUPADO,0.045,0.150,0.248,0.267
TRANSPORTE ESCOLAR,0.376,0.418,0.397,0.352


#### 4.6 Pruebas de arquitectura

In [67]:
# Define a flexible MLP architecture

class FlexibleModeChoiceMLP(nn.Module):

    def __init__(self, input_dim, num_classes, hidden_layers,
                 dropout_rates):

        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim, dropout_rate in zip(hidden_layers, dropout_rates):

            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))

            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# Define architecture configurations

nn_architectures = {
    "MLP_A_128_64_64": {
        "hidden_layers": [128, 64, 64],
        "dropout_rates": [0.20, 0.20, 0.30]
    },
    "MLP_B_64_32": {
        "hidden_layers": [64, 32],
        "dropout_rates": [0.20, 0.30]
    },
    "MLP_C_128_64": {
        "hidden_layers": [128, 64],
        "dropout_rates": [0.20, 0.30]
    },
    "MLP_D_256_128_64_32": {
        "hidden_layers": [256, 128, 64, 32],
        "dropout_rates": [0.20, 0.20, 0.25, 0.30]
    }
}

print("Architectures defined:")
for name, config in nn_architectures.items():
    print(f"{name}: {config['hidden_layers']}")

Architectures defined:
MLP_A_128_64_64: [128, 64, 64]
MLP_B_64_32: [64, 32]
MLP_C_128_64: [128, 64]
MLP_D_256_128_64_32: [256, 128, 64, 32]


In [68]:
# Train and compare the four MLP architectures

nn_architecture_results = []

class_weights = nn_weight_experiments[0.50]

for name, config in nn_architectures.items():

    print(f"\n{'='*55}")
    print(f"Training: {name}")
    print(f"{'='*55}")

    torch.manual_seed(42)
    np.random.seed(42)

    model = FlexibleModeChoiceMLP(
        input_dim=X_nn_train_tensor.shape[1],
        num_classes=len(target_classes),
        hidden_layers=config["hidden_layers"],
        dropout_rates=config["dropout_rates"]
    )

    model, history, best_epoch = fit_mlp(
        model,
        nn_train_loader,
        nn_val_loader,
        class_weights,
        learning_rate=0.001,
        weight_decay=0.0001,
        max_epochs=100,
        patience=10,
        device=device
    )

    model.eval()

    with torch.no_grad():
        logits = model(X_nn_val_tensor.to(device))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

    predictions = probabilities.argmax(axis=1)

    best_idx = best_epoch - 1

    nn_architecture_results.append({
        "Architecture": name,
        "Accuracy": accuracy_score(y_nn_val, predictions),
        "Macro-F1": f1_score(y_nn_val, predictions, average="macro"),
        "Log Loss": log_loss(
            y_nn_val,
            probabilities,
            labels=np.arange(len(target_classes))
        ),
        "Best Epoch": best_epoch,
        "Train Loss": history["train_loss"][best_idx],
        "Validation Loss": history["val_loss"][best_idx],
        "Model": model,
        "History": history,
        "Predictions": predictions,
        "Probabilities": probabilities
    })

nn_architecture_comparison = pd.DataFrame([
    {
        key: result[key]
        for key in [
            "Architecture", "Accuracy", "Macro-F1",
            "Log Loss", "Best Epoch", "Train Loss",
            "Validation Loss"
        ]
    }
    for result in nn_architecture_results
])

display(nn_architecture_comparison.round(4))


Training: MLP_A_128_64_64
Epoch 010 | Train Loss: 1.5675 | Validation Loss: 1.5080
Epoch 020 | Train Loss: 1.5325 | Validation Loss: 1.4866
Epoch 030 | Train Loss: 1.5106 | Validation Loss: 1.4783
Epoch 040 | Train Loss: 1.4996 | Validation Loss: 1.4756
Epoch 050 | Train Loss: 1.4907 | Validation Loss: 1.4730
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.4714

Training: MLP_B_64_32
Epoch 010 | Train Loss: 1.6085 | Validation Loss: 1.5304
Epoch 020 | Train Loss: 1.5769 | Validation Loss: 1.5077
Epoch 030 | Train Loss: 1.5613 | Validation Loss: 1.4970
Epoch 040 | Train Loss: 1.5536 | Validation Loss: 1.4911
Epoch 050 | Train Loss: 1.5462 | Validation Loss: 1.4892
Epoch 060 | Train Loss: 1.5444 | Validation Loss: 1.4857
Epoch 070 | Train Loss: 1.5395 | Validation Loss: 1.4846
Epoch 080 | Train Loss: 1.5362 | Validation Loss: 1.4836
Early stopping at epoch 85.
Best epoch: 75
Best validation loss: 1.4810

Training: MLP_C_128_64
Epoch 010 | Train Loss: 1.5573 | Validati

,Architecture,Accuracy,Macro-F1,Log Loss,Best Epoch,Train Loss,Validation Loss
0,MLP_A_128_64_64,0.5575,0.3647,1.3033,45,1.4984,1.4714
1,MLP_B_64_32,0.5578,0.3541,1.3118,75,1.5347,1.4810
2,MLP_C_128_64,0.5520,0.3700,1.2973,66,1.4755,1.4637
3,MLP_D_256_128_64_32,0.5526,0.3526,1.3162,38,1.5276,1.4858


In [69]:
# Inspect the saved cross-validation partitions

print("Available partition keys:")
for key in partitions.files:
    print(f"{key}: {partitions[key].shape}")

Available partition keys:
train_indices: (80114,)
test_indices: (20029,)
y_train: (80114,)
y_test: (20029,)
groups_train: (80114,)
groups_test: (20029,)
fold_1_train: (64091,)
fold_1_validation: (16023,)
fold_2_train: (64090,)
fold_2_validation: (16024,)
fold_3_train: (64091,)
fold_3_validation: (16023,)
fold_4_train: (64092,)
fold_4_validation: (16022,)
fold_5_train: (64092,)
fold_5_validation: (16022,)


In [70]:
# Validate the integrity of the five grouped CV folds

fold_validation_indices = []

for fold in range(1, 6):

    train_fold_idx = partitions[f"fold_{fold}_train"]
    val_fold_idx = partitions[f"fold_{fold}_validation"]

    train_groups = groups_train[train_fold_idx]
    val_groups = groups_train[val_fold_idx]

    group_overlap = np.intersect1d(
        np.unique(train_groups),
        np.unique(val_groups)
    )

    fold_validation_indices.extend(val_fold_idx.tolist())

    print(
        f"Fold {fold}: "
        f"Train = {len(train_fold_idx):,} | "
        f"Validation = {len(val_fold_idx):,} | "
        f"Group overlap = {len(group_overlap)}"
    )

validation_counts = pd.Series(fold_validation_indices).value_counts()

print("\nOverall validation integrity:")
print(f"Unique validation observations: {validation_counts.size:,}")
print(f"Repeated validation observations: {(validation_counts > 1).sum()}")
print(f"Missing validation observations: {80114 - validation_counts.size:,}")

Fold 1: Train = 64,091 | Validation = 16,023 | Group overlap = 0
Fold 2: Train = 64,090 | Validation = 16,024 | Group overlap = 0
Fold 3: Train = 64,091 | Validation = 16,023 | Group overlap = 0
Fold 4: Train = 64,092 | Validation = 16,022 | Group overlap = 0
Fold 5: Train = 64,092 | Validation = 16,022 | Group overlap = 0

Overall validation integrity:
Unique validation observations: 80,114
Repeated validation observations: 0
Missing validation observations: 0


In [71]:
# Calculate class weights using only the training observations

def get_alpha_class_weights(y, n_classes, alpha=0.50):

    class_counts = np.bincount(y, minlength=n_classes)

    assert np.all(class_counts > 0), "A class is missing from the training data."

    base_weights = len(y) / (n_classes * class_counts)

    weights = base_weights ** alpha
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)

In [72]:
# Train the MLP for a fixed number of epochs

def fit_mlp_fixed_epochs(
    model,
    train_loader,
    class_weights,
    epochs,
    learning_rate=0.001,
    weight_decay=0.0001,
    device=device
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device)
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    for epoch in range(epochs):
        train_mlp_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

    return model

In [73]:
# Prepare MLP data loaders with fold-specific preprocessing

def prepare_mlp_loaders(X_fit, y_fit, X_eval, y_eval, batch_size=512):

    preprocessor = clone(nn_preprocessor)

    X_fit_transformed = preprocessor.fit_transform(X_fit)
    X_eval_transformed = preprocessor.transform(X_eval)

    fit_dataset = TensorDataset(
        torch.tensor(X_fit_transformed, dtype=torch.float32),
        torch.tensor(y_fit, dtype=torch.long)
    )

    eval_dataset = TensorDataset(
        torch.tensor(X_eval_transformed, dtype=torch.float32),
        torch.tensor(y_eval, dtype=torch.long)
    )

    fit_loader = DataLoader(
        fit_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return preprocessor, fit_loader, eval_loader

### 4.7 Entrenamiento por K-fold para definir la arquitectura

In [74]:
# Prepare the inner training and validation split for Fold 1

from sklearn.model_selection import StratifiedGroupKFold

fold = 1

outer_train_idx = partitions[f"fold_{fold}_train"]
outer_val_idx = partitions[f"fold_{fold}_validation"]

X_outer_train = X_train.iloc[outer_train_idx].copy()
y_outer_train = y_train[outer_train_idx]
groups_outer_train = groups_train[outer_train_idx]

X_outer_val = X_train.iloc[outer_val_idx].copy()
y_outer_val = y_train[outer_val_idx]

inner_cv = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=100 + fold
)

inner_train_idx, inner_val_idx = next(
    inner_cv.split(X_outer_train, y_outer_train, groups_outer_train)
)

X_inner_train = X_outer_train.iloc[inner_train_idx].copy()
y_inner_train = y_outer_train[inner_train_idx]

X_inner_val = X_outer_train.iloc[inner_val_idx].copy()
y_inner_val = y_outer_train[inner_val_idx]

group_overlap = np.intersect1d(
    groups_outer_train[inner_train_idx],
    groups_outer_train[inner_val_idx]
)

print(f"Outer training observations: {len(X_outer_train):,}")
print(f"Outer validation observations: {len(X_outer_val):,}")
print(f"Inner training observations: {len(X_inner_train):,}")
print(f"Inner validation observations: {len(X_inner_val):,}")
print(f"Inner group overlap: {len(group_overlap)}")

Outer training observations: 64,091
Outer validation observations: 16,023
Inner training observations: 54,935
Inner validation observations: 9,156
Inner group overlap: 0


#### Particion 1

In [75]:
# Train and compare MLP architectures on the inner split of Fold 1

torch.manual_seed(101)

nn_preprocessor_fold1, nn_train_loader, nn_val_loader = prepare_mlp_loaders(
    X_inner_train, y_inner_train,
    X_inner_val, y_inner_val
)

nn_class_weights_fold1 = get_alpha_class_weights(
    y_inner_train,
    n_classes=len(target_classes),
    alpha=0.50
)

input_dim = nn_preprocessor_fold1.transform(
    X_inner_train.iloc[:1]
).shape[1]

fold1_architecture_results = {}

for name in ["MLP_A_128_64_64", "MLP_C_128_64"]:

    print(f"\nTraining {name}...")

    torch.manual_seed(101)

    config = nn_architectures[name]

    model = FlexibleModeChoiceMLP(
        input_dim=input_dim,
        num_classes=len(target_classes),
        hidden_layers=config["hidden_layers"],
        dropout_rates=config["dropout_rates"]
    )

    model, history, best_epoch = fit_mlp(
        model,
        nn_train_loader,
        nn_val_loader,
        nn_class_weights_fold1,
        learning_rate=0.001,
        weight_decay=0.0001,
        max_epochs=100,
        patience=10,
        device=device
    )

    model.eval()
    probabilities = []

    with torch.no_grad():
        for batch_X, _ in nn_val_loader:
            logits = model(batch_X.to(device))
            probabilities.extend(
                torch.softmax(logits, dim=1).cpu().numpy()
            )

    probabilities = np.array(probabilities)
    predictions = probabilities.argmax(axis=1)

    fold1_architecture_results[name] = {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "accuracy": accuracy_score(y_inner_val, predictions),
        "macro_f1": f1_score(
            y_inner_val, predictions, average="macro", zero_division=0
        ),
        "log_loss": log_loss(
            y_inner_val, probabilities, labels=np.arange(len(target_classes))
        )
    }

    print(
        f"Accuracy: {fold1_architecture_results[name]['accuracy']:.4f} | "
        f"Macro-F1: {fold1_architecture_results[name]['macro_f1']:.4f} | "
        f"LogLoss: {fold1_architecture_results[name]['log_loss']:.4f} | "
        f"Best epoch: {best_epoch}"
    )


Training MLP_A_128_64_64...
Epoch 010 | Train Loss: 1.5732 | Validation Loss: 1.5268
Epoch 020 | Train Loss: 1.5283 | Validation Loss: 1.5019
Epoch 030 | Train Loss: 1.5143 | Validation Loss: 1.4944
Epoch 040 | Train Loss: 1.5021 | Validation Loss: 1.4894
Epoch 050 | Train Loss: 1.4874 | Validation Loss: 1.4882
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.4861
Accuracy: 0.5460 | Macro-F1: 0.3575 | LogLoss: 1.3063 | Best epoch: 45

Training MLP_C_128_64...
Epoch 010 | Train Loss: 1.5631 | Validation Loss: 1.5285
Epoch 020 | Train Loss: 1.5273 | Validation Loss: 1.5061
Epoch 030 | Train Loss: 1.5044 | Validation Loss: 1.4946
Epoch 040 | Train Loss: 1.4934 | Validation Loss: 1.4915
Epoch 050 | Train Loss: 1.4833 | Validation Loss: 1.4872
Epoch 060 | Train Loss: 1.4721 | Validation Loss: 1.4842
Epoch 070 | Train Loss: 1.4685 | Validation Loss: 1.4823
Epoch 080 | Train Loss: 1.4614 | Validation Loss: 1.4821
Early stopping at epoch 85.
Best epoch: 75
Best validation lo

In [76]:
# Refit both architectures on the complete outer training set

torch.manual_seed(101)

nn_preprocessor_outer, outer_train_loader, outer_val_loader = prepare_mlp_loaders(
    X_outer_train, y_outer_train,
    X_outer_val, y_outer_val
)

nn_class_weights_outer = get_alpha_class_weights(
    y_outer_train,
    n_classes=len(target_classes),
    alpha=0.50
)

input_dim = nn_preprocessor_outer.transform(
    X_outer_train.iloc[:1]
).shape[1]

fold1_outer_results = {}

for name in ["MLP_A_128_64_64", "MLP_C_128_64"]:

    print(f"\nEvaluating {name} on outer validation...")

    torch.manual_seed(101)

    config = nn_architectures[name]
    best_epoch = fold1_architecture_results[name]["best_epoch"]

    model = FlexibleModeChoiceMLP(
        input_dim=input_dim,
        num_classes=len(target_classes),
        hidden_layers=config["hidden_layers"],
        dropout_rates=config["dropout_rates"]
    )

    model = fit_mlp_fixed_epochs(
        model,
        outer_train_loader,
        nn_class_weights_outer,
        epochs=best_epoch,
        learning_rate=0.001,
        weight_decay=0.0001,
        device=device
    )

    model.eval()
    probabilities = []

    with torch.no_grad():
        for batch_X, _ in outer_val_loader:
            logits = model(batch_X.to(device))
            probabilities.extend(
                torch.softmax(logits, dim=1).cpu().numpy()
            )

    probabilities = np.array(probabilities)
    predictions = probabilities.argmax(axis=1)

    fold1_outer_results[name] = {
        "model": model,
        "probabilities": probabilities,
        "predictions": predictions,
        "accuracy": accuracy_score(y_outer_val, predictions),
        "macro_f1": f1_score(
            y_outer_val, predictions, average="macro", zero_division=0
        ),
        "log_loss": log_loss(
            y_outer_val, probabilities,
            labels=np.arange(len(target_classes))
        )
    }

    print(
        f"Accuracy: {fold1_outer_results[name]['accuracy']:.4f} | "
        f"Macro-F1: {fold1_outer_results[name]['macro_f1']:.4f} | "
        f"LogLoss: {fold1_outer_results[name]['log_loss']:.4f}"
    )


Evaluating MLP_A_128_64_64 on outer validation...
Accuracy: 0.5394 | Macro-F1: 0.3516 | LogLoss: 1.3333

Evaluating MLP_C_128_64 on outer validation...
Accuracy: 0.5468 | Macro-F1: 0.3488 | LogLoss: 1.3167


#### particion 2

In [77]:
# Prepare the inner training and validation split for Fold 2

fold = 2

outer_train_idx = partitions[f"fold_{fold}_train"]
outer_val_idx = partitions[f"fold_{fold}_validation"]

X_outer_train = X_train.iloc[outer_train_idx].copy()
y_outer_train = y_train[outer_train_idx]
groups_outer_train = groups_train[outer_train_idx]

X_outer_val = X_train.iloc[outer_val_idx].copy()
y_outer_val = y_train[outer_val_idx]

inner_cv = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=100 + fold
)

inner_train_idx, inner_val_idx = next(
    inner_cv.split(X_outer_train, y_outer_train, groups_outer_train)
)

X_inner_train = X_outer_train.iloc[inner_train_idx].copy()
y_inner_train = y_outer_train[inner_train_idx]

X_inner_val = X_outer_train.iloc[inner_val_idx].copy()
y_inner_val = y_outer_train[inner_val_idx]

group_overlap = np.intersect1d(
    groups_outer_train[inner_train_idx],
    groups_outer_train[inner_val_idx]
)

print(f"Outer training observations: {len(X_outer_train):,}")
print(f"Outer validation observations: {len(X_outer_val):,}")
print(f"Inner training observations: {len(X_inner_train):,}")
print(f"Inner validation observations: {len(X_inner_val):,}")
print(f"Inner group overlap: {len(group_overlap)}")

Outer training observations: 64,090
Outer validation observations: 16,024
Inner training observations: 54,935
Inner validation observations: 9,155
Inner group overlap: 0


In [78]:
# Run the complete nested validation process for one fold

def run_mlp_fold(fold):

    outer_train_idx = partitions[f"fold_{fold}_train"]
    outer_val_idx = partitions[f"fold_{fold}_validation"]

    X_outer_train = X_train.iloc[outer_train_idx].copy()
    y_outer_train = y_train[outer_train_idx]
    groups_outer_train = groups_train[outer_train_idx]

    X_outer_val = X_train.iloc[outer_val_idx].copy()
    y_outer_val = y_train[outer_val_idx]

    inner_cv = StratifiedGroupKFold(
        n_splits=7,
        shuffle=True,
        random_state=100 + fold
    )

    inner_train_idx, inner_val_idx = next(
        inner_cv.split(
            X_outer_train,
            y_outer_train,
            groups_outer_train
        )
    )

    X_inner_train = X_outer_train.iloc[inner_train_idx].copy()
    y_inner_train = y_outer_train[inner_train_idx]

    X_inner_val = X_outer_train.iloc[inner_val_idx].copy()
    y_inner_val = y_outer_train[inner_val_idx]

    overlap = np.intersect1d(
        groups_outer_train[inner_train_idx],
        groups_outer_train[inner_val_idx]
    )

    assert len(overlap) == 0, "Group leakage detected."

    print(f"\n{'='*55}")
    print(f"FOLD {fold}")
    print(f"{'='*55}")

    # Prepare inner data
    _, inner_train_loader, inner_val_loader = prepare_mlp_loaders(
        X_inner_train, y_inner_train,
        X_inner_val, y_inner_val
    )

    inner_weights = get_alpha_class_weights(
        y_inner_train, len(target_classes), alpha=0.50
    )

    # Prepare outer data
    outer_preprocessor, outer_train_loader, outer_val_loader = prepare_mlp_loaders(
        X_outer_train, y_outer_train,
        X_outer_val, y_outer_val
    )

    outer_weights = get_alpha_class_weights(
        y_outer_train, len(target_classes), alpha=0.50
    )

    input_dim = outer_preprocessor.transform(
        X_outer_train.iloc[:1]
    ).shape[1]

    fold_results = {}

    for name in ["MLP_A_128_64_64", "MLP_C_128_64"]:

        print(f"\nTraining {name}...")

        config = nn_architectures[name]

        # Inner training: select the best epoch
        torch.manual_seed(1000 + fold)

        model = FlexibleModeChoiceMLP(
            input_dim=input_dim,
            num_classes=len(target_classes),
            hidden_layers=config["hidden_layers"],
            dropout_rates=config["dropout_rates"]
        )

        model, history, best_epoch = fit_mlp(
            model,
            inner_train_loader,
            inner_val_loader,
            inner_weights,
            learning_rate=0.001,
            weight_decay=0.0001,
            max_epochs=100,
            patience=10,
            device=device
        )

        # Outer training: refit using all outer training observations
        torch.manual_seed(1000 + fold)

        final_model = FlexibleModeChoiceMLP(
            input_dim=input_dim,
            num_classes=len(target_classes),
            hidden_layers=config["hidden_layers"],
            dropout_rates=config["dropout_rates"]
        )

        final_model = fit_mlp_fixed_epochs(
            final_model,
            outer_train_loader,
            outer_weights,
            epochs=best_epoch,
            learning_rate=0.001,
            weight_decay=0.0001,
            device=device
        )

        # Outer validation
        final_model.eval()
        probabilities = []

        with torch.no_grad():
            for batch_X, _ in outer_val_loader:
                logits = final_model(batch_X.to(device))
                probabilities.extend(
                    torch.softmax(logits, dim=1).cpu().numpy()
                )

        probabilities = np.array(probabilities)
        predictions = probabilities.argmax(axis=1)

        fold_results[name] = {
            "best_epoch": best_epoch,
            "accuracy": accuracy_score(y_outer_val, predictions),
            "macro_f1": f1_score(
                y_outer_val, predictions,
                average="macro", zero_division=0
            ),
            "log_loss": log_loss(
                y_outer_val,
                probabilities,
                labels=np.arange(len(target_classes))
            ),
            "probabilities": probabilities,
            "predictions": predictions
        }

        print(
            f"Best epoch: {best_epoch} | "
            f"Accuracy: {fold_results[name]['accuracy']:.4f} | "
            f"Macro-F1: {fold_results[name]['macro_f1']:.4f} | "
            f"LogLoss: {fold_results[name]['log_loss']:.4f}"
        )

    return fold_results

In [79]:
# Run nested validation for Fold 2

fold2_results = run_mlp_fold(2)


FOLD 2

Training MLP_A_128_64_64...
Epoch 010 | Train Loss: 1.5769 | Validation Loss: 1.5344
Epoch 020 | Train Loss: 1.5339 | Validation Loss: 1.5136
Epoch 030 | Train Loss: 1.5174 | Validation Loss: 1.5032
Epoch 040 | Train Loss: 1.5070 | Validation Loss: 1.5030
Epoch 050 | Train Loss: 1.4969 | Validation Loss: 1.4997
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.4990
Best epoch: 45 | Accuracy: 0.5516 | Macro-F1: 0.3617 | LogLoss: 1.3037

Training MLP_C_128_64...
Epoch 010 | Train Loss: 1.5619 | Validation Loss: 1.5315
Epoch 020 | Train Loss: 1.5275 | Validation Loss: 1.5124
Epoch 030 | Train Loss: 1.5088 | Validation Loss: 1.5023
Epoch 040 | Train Loss: 1.4998 | Validation Loss: 1.5024
Early stopping at epoch 48.
Best epoch: 38
Best validation loss: 1.4980
Best epoch: 38 | Accuracy: 0.5519 | Macro-F1: 0.3583 | LogLoss: 1.3047


In [80]:
# Run nested validation for Fold 3

fold3_results = run_mlp_fold(3)


FOLD 3

Training MLP_A_128_64_64...
Epoch 010 | Train Loss: 1.5712 | Validation Loss: 1.5380
Epoch 020 | Train Loss: 1.5350 | Validation Loss: 1.5167
Epoch 030 | Train Loss: 1.5144 | Validation Loss: 1.5046
Epoch 040 | Train Loss: 1.5022 | Validation Loss: 1.5024
Epoch 050 | Train Loss: 1.4970 | Validation Loss: 1.4991
Epoch 060 | Train Loss: 1.4860 | Validation Loss: 1.4997
Epoch 070 | Train Loss: 1.4753 | Validation Loss: 1.5008
Early stopping at epoch 73.
Best epoch: 63
Best validation loss: 1.4968
Best epoch: 63 | Accuracy: 0.5498 | Macro-F1: 0.3551 | LogLoss: 1.3111

Training MLP_C_128_64...
Epoch 010 | Train Loss: 1.5588 | Validation Loss: 1.5326
Epoch 020 | Train Loss: 1.5244 | Validation Loss: 1.5148
Epoch 030 | Train Loss: 1.5095 | Validation Loss: 1.5039
Epoch 040 | Train Loss: 1.4955 | Validation Loss: 1.4994
Epoch 050 | Train Loss: 1.4868 | Validation Loss: 1.4991
Epoch 060 | Train Loss: 1.4770 | Validation Loss: 1.4974
Early stopping at epoch 68.
Best epoch: 58
Best valid

In [81]:
# Run nested validation for Fold 4

fold4_results = run_mlp_fold(4)


FOLD 4

Training MLP_A_128_64_64...
Epoch 010 | Train Loss: 1.5621 | Validation Loss: 1.5149
Epoch 020 | Train Loss: 1.5276 | Validation Loss: 1.4984
Epoch 030 | Train Loss: 1.5089 | Validation Loss: 1.4915
Epoch 040 | Train Loss: 1.4965 | Validation Loss: 1.4916
Epoch 050 | Train Loss: 1.4860 | Validation Loss: 1.4881
Early stopping at epoch 55.
Best epoch: 45
Best validation loss: 1.4859
Best epoch: 45 | Accuracy: 0.5441 | Macro-F1: 0.3587 | LogLoss: 1.3220

Training MLP_C_128_64...
Epoch 010 | Train Loss: 1.5641 | Validation Loss: 1.5187
Epoch 020 | Train Loss: 1.5270 | Validation Loss: 1.4992
Epoch 030 | Train Loss: 1.5089 | Validation Loss: 1.4935
Epoch 040 | Train Loss: 1.4944 | Validation Loss: 1.4899
Epoch 050 | Train Loss: 1.4855 | Validation Loss: 1.4886
Epoch 060 | Train Loss: 1.4786 | Validation Loss: 1.4883
Epoch 070 | Train Loss: 1.4731 | Validation Loss: 1.4855
Early stopping at epoch 79.
Best epoch: 69
Best validation loss: 1.4851
Best epoch: 69 | Accuracy: 0.5507 | Ma

In [82]:
# Run nested validation for Fold 5

fold5_results = run_mlp_fold(5)


FOLD 5

Training MLP_A_128_64_64...
Epoch 010 | Train Loss: 1.5713 | Validation Loss: 1.5453
Epoch 020 | Train Loss: 1.5358 | Validation Loss: 1.5235
Epoch 030 | Train Loss: 1.5155 | Validation Loss: 1.5150
Epoch 040 | Train Loss: 1.4996 | Validation Loss: 1.5104
Epoch 050 | Train Loss: 1.4878 | Validation Loss: 1.5112
Epoch 060 | Train Loss: 1.4850 | Validation Loss: 1.5077
Epoch 070 | Train Loss: 1.4727 | Validation Loss: 1.5078
Early stopping at epoch 75.
Best epoch: 65
Best validation loss: 1.5059
Best epoch: 65 | Accuracy: 0.5416 | Macro-F1: 0.3540 | LogLoss: 1.3161

Training MLP_C_128_64...
Epoch 010 | Train Loss: 1.5566 | Validation Loss: 1.5397
Epoch 020 | Train Loss: 1.5250 | Validation Loss: 1.5220
Epoch 030 | Train Loss: 1.5067 | Validation Loss: 1.5128
Epoch 040 | Train Loss: 1.4937 | Validation Loss: 1.5108
Epoch 050 | Train Loss: 1.4852 | Validation Loss: 1.5076
Epoch 060 | Train Loss: 1.4790 | Validation Loss: 1.5062
Epoch 070 | Train Loss: 1.4708 | Validation Loss: 1.5

### 4.8 Resultados CV Aquitectura

In [83]:
# Consolidate MLP architecture comparison results

fold_results_dict = {
    1: fold1_outer_results,
    2: fold2_results,
    3: fold3_results,
    4: fold4_results,
    5: fold5_results
}

records = []

for fold, results in fold_results_dict.items():

    for architecture, metrics in results.items():

        records.append({
            "fold": fold,
            "architecture": architecture,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "log_loss": metrics["log_loss"]
        })

mlp_architecture_results = pd.DataFrame(records)

mlp_architecture_summary = (
    mlp_architecture_results
    .groupby("architecture")
    .agg({
        "accuracy": ["mean", "std"],
        "macro_f1": ["mean", "std"],
        "log_loss": ["mean", "std"]
    })
    .round(5)
)

display(mlp_architecture_results)
display(mlp_architecture_summary)

mlp_architecture_results.to_csv(
    ml_path / "MLP_architecture_CV_results.csv",
    index=False
)

mlp_architecture_summary.to_csv(
    ml_path / "MLP_architecture_CV_summary.csv"
)

print("Architecture comparison results saved successfully.")

,fold,architecture,accuracy,macro_f1,log_loss
0,1,MLP_A_128_64_64,0.539412,0.351632,1.333320
1,1,MLP_C_128_64,0.546839,0.348785,1.316748
2,2,MLP_A_128_64_64,0.551610,0.361688,1.303730
3,2,MLP_C_128_64,0.551860,0.358278,1.304660
4,3,MLP_A_128_64_64,0.549835,0.355141,1.311130
5,3,MLP_C_128_64,0.550334,0.357775,1.317909
6,4,MLP_A_128_64_64,0.544064,0.358657,1.321971
7,4,MLP_C_128_64,0.550680,0.354725,1.316451
8,5,MLP_A_128_64_64,0.541630,0.354002,1.316142
9,5,MLP_C_128_64,0.546311,0.359819,1.313651


accuracy          macro_f1          log_loss         
                    mean      std     mean      std     mean      std
architecture                                                         
MLP_A_128_64_64  0.54531  0.00525  0.35622  0.00397  1.31726  0.01120
MLP_C_128_64     0.54920  0.00247  0.35588  0.00437  1.31388  0.00539

Architecture comparison results saved successfully.


### 4.8 Seleccion Learning rate y optimizador

In [84]:
# Train MLP with configurable optimizer and early stopping

def fit_mlp_optimizer(
    model,
    train_loader,
    val_loader,
    class_weights,
    optimizer_name="Adam",
    learning_rate=0.001,
    weight_decay=0.0001,
    max_epochs=100,
    patience=10,
    device=device
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device)
    )

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
    else:
        raise ValueError("Unsupported optimizer.")

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(max_epochs):

        train_loss = train_mlp_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss = evaluate_mlp_loss(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1:03d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Validation Loss: {val_loss:.4f}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}.")
            break

    model.load_state_dict(best_state)

    return model, history, best_epoch

In [85]:
# Compare six MLP optimizer and learning-rate configurations

# Recreate the preselection data loaders
nn_preprocessor_opt, nn_train_loader_opt, nn_val_loader_opt = prepare_mlp_loaders(
    X_nn_train, y_nn_train,
    X_nn_val, y_nn_val,
    batch_size=512
)

nn_class_weights_opt = get_alpha_class_weights(
    y_nn_train,
    n_classes=len(target_classes),
    alpha=0.50
)

input_dim = nn_preprocessor_opt.transform(
    X_nn_train.iloc[:1]
).shape[1]

optimizer_configs = [
    {"name": "ANN_1", "optimizer": "Adam", "lr": 0.0005},
    {"name": "ANN_2", "optimizer": "Adam", "lr": 0.001},
    {"name": "ANN_3", "optimizer": "Adam", "lr": 0.002},
    {"name": "ANN_4", "optimizer": "AdamW", "lr": 0.0005},
    {"name": "ANN_5", "optimizer": "AdamW", "lr": 0.001},
    {"name": "ANN_6", "optimizer": "AdamW", "lr": 0.002}
]

ann_optimization_results = {}
ann_optimization_records = []

for config in optimizer_configs:

    name = config["name"]

    print(f"\n{'='*50}")
    print(f"Training {name}: {config['optimizer']} | LR={config['lr']}")
    print(f"{'='*50}")

    torch.manual_seed(42)

    model = FlexibleModeChoiceMLP(
        input_dim=input_dim,
        num_classes=len(target_classes),
        hidden_layers=[128, 64],
        dropout_rates=[0.20, 0.30]
    )

    model, history, best_epoch = fit_mlp_optimizer(
        model,
        nn_train_loader_opt,
        nn_val_loader_opt,
        nn_class_weights_opt,
        optimizer_name=config["optimizer"],
        learning_rate=config["lr"],
        weight_decay=0.0001,
        max_epochs=100,
        patience=10,
        device=device
    )

    model.eval()
    probabilities = []

    with torch.no_grad():
        for batch_X, _ in nn_val_loader_opt:
            logits = model(batch_X.to(device))
            probabilities.extend(
                torch.softmax(logits, dim=1).cpu().numpy()
            )

    probabilities = np.array(probabilities)
    predictions = probabilities.argmax(axis=1)

    metrics = {
        "name": name,
        "optimizer": config["optimizer"],
        "learning_rate": config["lr"],
        "best_epoch": best_epoch,
        "accuracy": accuracy_score(y_nn_val, predictions),
        "macro_f1": f1_score(
            y_nn_val, predictions,
            average="macro", zero_division=0
        ),
        "log_loss": log_loss(
            y_nn_val,
            probabilities,
            labels=np.arange(len(target_classes))
        )
    }

    ann_optimization_results[name] = {
        "model": model,
        "history": history,
        "probabilities": probabilities,
        "predictions": predictions,
        "metrics": metrics
    }

    ann_optimization_records.append(metrics)

    print(
        f"Best epoch: {best_epoch} | "
        f"Accuracy: {metrics['accuracy']:.4f} | "
        f"Macro-F1: {metrics['macro_f1']:.4f} | "
        f"LogLoss: {metrics['log_loss']:.4f}"
    )

ann_optimization_df = pd.DataFrame(ann_optimization_records)

ann_optimization_df.to_csv(
    ml_path / "ANN_optimizer_learning_rate_comparison.csv",
    index=False
)

print("\nAll six ANN experiments completed and saved.")


Training ANN_1: Adam | LR=0.0005
Epoch 010 | Train Loss: 1.5905 | Validation Loss: 1.5268
Epoch 020 | Train Loss: 1.5479 | Validation Loss: 1.4983
Epoch 030 | Train Loss: 1.5297 | Validation Loss: 1.4864
Epoch 040 | Train Loss: 1.5143 | Validation Loss: 1.4806
Epoch 050 | Train Loss: 1.5085 | Validation Loss: 1.4749
Epoch 060 | Train Loss: 1.4984 | Validation Loss: 1.4717
Epoch 070 | Train Loss: 1.4950 | Validation Loss: 1.4698
Early stopping at epoch 76.
Best epoch: 66 | Accuracy: 0.5516 | Macro-F1: 0.3656 | LogLoss: 1.3005

Training ANN_2: Adam | LR=0.001
Epoch 010 | Train Loss: 1.5573 | Validation Loss: 1.5022
Epoch 020 | Train Loss: 1.5208 | Validation Loss: 1.4813
Epoch 030 | Train Loss: 1.5059 | Validation Loss: 1.4738
Epoch 040 | Train Loss: 1.4903 | Validation Loss: 1.4719
Epoch 050 | Train Loss: 1.4888 | Validation Loss: 1.4682
Epoch 060 | Train Loss: 1.4787 | Validation Loss: 1.4673
Epoch 070 | Train Loss: 1.4744 | Validation Loss: 1.4661
Early stopping at epoch 76.
Best epo

### Selected ANN Configuration

**Model architecture:** Multilayer Perceptron (MLP) with two hidden layers (128 and 64 neurons) and 11 output classes.

**Activation function:** ReLU.

**Dropout:** 0.20 in the first hidden layer and 0.30 in the second hidden layer.

**Optimizer:** AdamW.

**Learning rate:** 0.0005.

**Weight decay:** 0.0001.

**Class weighting:** Inverse-frequency class weights with α = 0.50.

**Batch size:** 512.

**Training epochs:** 66 epochs, selected through early stopping during model optimization.

**Model selection criterion:** Macro-F1 as the primary metric, with Log Loss and Accuracy as complementary evaluation metrics.

### 4.9 Entrenamiento final y evaluacion en test

In [86]:
# Final ANN training and independent test evaluation

# Fit preprocessing using the complete training set
ann_final_preprocessor = clone(nn_preprocessor)

X_final_transformed = ann_final_preprocessor.fit_transform(X_train)
X_test_transformed = ann_final_preprocessor.transform(X_test)

# Create data loaders
final_train_dataset = TensorDataset(
    torch.tensor(X_final_transformed, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

final_test_dataset = TensorDataset(
    torch.tensor(X_test_transformed, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=512,
    shuffle=True
)

final_test_loader = DataLoader(
    final_test_dataset,
    batch_size=512,
    shuffle=False
)

# Calculate class weights using the complete training set
final_class_weights = get_alpha_class_weights(
    y_train,
    n_classes=len(target_classes),
    alpha=0.50
)

# Initialize the selected architecture
torch.manual_seed(42)

ann_final_model = FlexibleModeChoiceMLP(
    input_dim=X_final_transformed.shape[1],
    num_classes=len(target_classes),
    hidden_layers=[128, 64],
    dropout_rates=[0.20, 0.30]
).to(device)

# Configure final optimizer
criterion = nn.CrossEntropyLoss(
    weight=final_class_weights.to(device)
)

optimizer = torch.optim.AdamW(
    ann_final_model.parameters(),
    lr=0.0005,
    weight_decay=0.0001
)

# Train for the selected number of epochs
for epoch in range(66):

    train_loss = train_mlp_epoch(
        ann_final_model,
        final_train_loader,
        optimizer,
        criterion,
        device
    )

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:02d}/66 | Loss: {train_loss:.4f}")

# Evaluate on the untouched test set
ann_final_model.eval()
test_probabilities = []

with torch.no_grad():
    for batch_X, _ in final_test_loader:
        logits = ann_final_model(batch_X.to(device))
        test_probabilities.extend(
            torch.softmax(logits, dim=1).cpu().numpy()
        )

test_probabilities = np.array(test_probabilities)
test_predictions = test_probabilities.argmax(axis=1)

ann_final_metrics = {
    "accuracy": accuracy_score(y_test, test_predictions),
    "macro_f1": f1_score(
        y_test, test_predictions,
        average="macro", zero_division=0
    ),
    "log_loss": log_loss(
        y_test,
        test_probabilities,
        labels=np.arange(len(target_classes))
    )
}

print("\nFINAL ANN TEST RESULTS")
for metric, value in ann_final_metrics.items():
    print(f"{metric}: {value:.4f}")

Epoch 10/66 | Loss: 1.5756
Epoch 20/66 | Loss: 1.5375
Epoch 30/66 | Loss: 1.5239
Epoch 40/66 | Loss: 1.5077
Epoch 50/66 | Loss: 1.4972
Epoch 60/66 | Loss: 1.4949

FINAL ANN TEST RESULTS
accuracy: 0.5516
macro_f1: 0.3654
log_loss: 1.2872


In [87]:
import json
import joblib
import torch

# Save final ANN model weights
torch.save(
    ann_final_model.state_dict(),
    ml_path / "ANN_final_model.pth"
)

# Save fitted preprocessing pipeline
joblib.dump(
    ann_final_preprocessor,
    ml_path / "ANN_final_preprocessor.joblib"
)

# Save model configuration and evaluation results
ann_metadata = {
    "model": "Multilayer Perceptron",
    "architecture": [128, 64],
    "output_classes": len(target_classes),
    "activation": "ReLU",
    "dropout_rates": [0.20, 0.30],
    "optimizer": "AdamW",
    "learning_rate": 0.0005,
    "weight_decay": 0.0001,
    "class_weight_alpha": 0.50,
    "batch_size": 512,
    "epochs": 66,
    "input_dim": int(ann_final_preprocessor.transform(X_train.iloc[:1]).shape[1]),
    "features": b3_features,
    "target_classes": target_classes,
    "test_metrics": {
        key: float(value)
        for key, value in ann_final_metrics.items()
    }
}

with open(ml_path / "ANN_final_metadata.json", "w") as file:
    json.dump(ann_metadata, file, indent=4)

print("ANN artifacts saved successfully.")
print("Files:")
print("- ANN_final_model.pth")
print("- ANN_final_preprocessor.joblib")
print("- ANN_final_metadata.json")

ANN artifacts saved successfully.
Files:
- ANN_final_model.pth
- ANN_final_preprocessor.joblib
- ANN_final_metadata.json
